# 🚀 FictiPay Churn Prediction — Full Pipeline

**Architecture:** Memory-Safe Streaming Feature Engineering → Model Zoo (6 models × 10-fold CV) → Stacking Ensemble → `predictions.csv`

**Requirements:**
- Runtime → Change runtime type → **T4 GPU**
- Dataset in Google Drive root: `bkash-presents-nsucec-datathon/`

---

## 📦 Step 0: Setup — Mount Drive, Install Dependencies, Verify GPU

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Verify dataset path exists
import os
BASE_PATH = "/content/drive/MyDrive/bkash-presents-nsucec-datathon/public"
assert os.path.exists(BASE_PATH), f"Dataset not found at {BASE_PATH}! Check your Drive."
print(f"✅ Dataset found at: {BASE_PATH}")
print(f"   Contents: {os.listdir(BASE_PATH)}")

In [ ]:
# Install required packages (Colab has most pre-installed, but we need these)
!pip install -q lightgbm xgboost catboost pyarrow --upgrade

# Verify GPU availability
import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print(f"✅ GPU detected: {gpu_name}")
    print(f"   CUDA version: {torch.version.cuda}")
    print(f"   GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("⚠️ No GPU detected! Go to Runtime → Change runtime type → T4 GPU")

In [ ]:
# ─── Core Imports ───
import gc
import glob
import time
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, f1_score, recall_score, precision_score, brier_score_loss, roc_curve
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.isotonic import IsotonicRegression
from scipy.stats import rankdata

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

print(f"LightGBM: {lgb.__version__}")
print(f"XGBoost:  {xgb.__version__}")
print(f"NumPy:    {np.__version__}")
print(f"Pandas:   {pd.__version__}")
print("\n✅ All imports successful.")

In [ ]:
# ─── Global Configuration ───
base_path = BASE_PATH

# Output directories (inside Colab's local filesystem for speed)
FEATURE_STORE_DIR = "/content/feature_store"
PROCESSED_DIR     = "/content/processed_data"
PREDICTIONS_DIR   = "/content/predictions"
MODELS_DIR        = "/content/models"
PLOTS_DIR         = "/content/plots"

for d in [FEATURE_STORE_DIR, PROCESSED_DIR, PREDICTIONS_DIR, MODELS_DIR, PLOTS_DIR]:
    os.makedirs(d, exist_ok=True)

# Reference dates
REF_DATE_RECENCY = pd.Timestamp("2024-03-31")
REF_DATE_MARCH   = pd.Timestamp("2024-03-31 23:59:59")
REF_DATE_RECENCY_NS = REF_DATE_RECENCY.value
REF_DATE_MARCH_NS   = REF_DATE_MARCH.value
NS_PER_DAY = 86400 * 10**9

# Transaction types
TRX_TYPES = ["P2P", "MerchantPay", "BillPay", "CashIn", "CashOut"]
TYPE_TO_INT = {t: i for i, t in enumerate(TRX_TYPES)}
OUTBOUND_TYPES = ["P2P", "MerchantPay", "BillPay", "CashOut"]
MONTHS = ["jan", "feb", "march"]

# Training constants
N_SPLITS = 10
RANDOM_STATE = 42

print("✅ Configuration set.")

---
## 🔧 Step 1: Feature Engineering (Memory-Safe Streaming)

This section processes 73M+ transaction rows and 77M+ balance rows **without** loading them fully into RAM.

### 1.0 — Build ID Mapping

In [ ]:
# ─── Build ID Mapping (string → int64 for 12× memory reduction) ───
print("Loading target IDs...")
train_labels = pd.read_csv(os.path.join(base_path, "train_labels.csv"))
test_ids_df  = pd.read_csv(os.path.join(base_path, "test.csv"))

train_sampled = train_labels
test_sampled  = test_ids_df

target_ids_list = sorted(set(train_sampled["ACCOUNT_ID"]) | set(test_sampled["ACCOUNT_ID"]))
id_to_int = {aid: i for i, aid in enumerate(target_ids_list)}
int_to_id = {i: aid for aid, i in id_to_int.items()}
n = len(target_ids_list)

print(f"✅ Total target customers: {n:,}")
print(f"   Train: {len(train_sampled):,}, Test: {len(test_sampled):,}")

### 1.1 — KYC Features

In [ ]:
# ─── Step 1: KYC Features ───
print("\n[Step 1/6] Processing KYC features...")
kyc = pd.read_parquet(os.path.join(base_path, "kyc.parquet"))

kyc["aid_int"] = kyc["ACCOUNT_ID"].map(id_to_int)
kyc = kyc.dropna(subset=["aid_int"]).copy()
kyc["aid_int"] = kyc["aid_int"].astype(int)

kyc["tenure_days"] = (REF_DATE_RECENCY - pd.to_datetime(kyc["ACCOUNT_OPEN_DATE"])).dt.days
kyc["GENDER"] = kyc["GENDER"].fillna("Unknown")
kyc["REGION"] = kyc["REGION"].fillna("Unknown")
kyc = pd.get_dummies(kyc, columns=["GENDER", "REGION"], prefix=["gender", "region"], dtype=float)

drop_cols = ["ACCOUNT_ID", "ACCOUNT_TYPE", "ACCOUNT_OPEN_DATE"]
kyc = kyc.drop(columns=[c for c in drop_cols if c in kyc.columns])
kyc = kyc.set_index("aid_int").sort_index()
kyc = kyc.reindex(range(n), fill_value=0)

kyc.to_parquet(os.path.join(FEATURE_STORE_DIR, "kyc_features.parquet"))
print(f"  ✅ KYC features: {kyc.shape[1]} columns written.")
del kyc; gc.collect()

### 1.2 — Transaction Features (Single-Pass Per File)

In [ ]:
# ─── Step 2: Streaming Transaction Features ───
print("\n[Step 2/6] Streaming transaction features (single-pass per file)...")
trx_files = sorted(glob.glob(os.path.join(base_path, "transactions", "*.parquet")))
print(f"  Found {len(trx_files)} transaction files.")

# Global recency accumulator
global_last_dt_ns = np.full(n, -1, dtype=np.int64)

for month_idx, filepath in enumerate(trx_files):
    if month_idx >= len(MONTHS):
        break
    month = MONTHS[month_idx]
    is_march = (month == "march")
    t0 = time.time()
    print(f"\n  Processing {month} transactions ({os.path.basename(filepath)})...")

    # Monthly accumulators
    out_count  = np.zeros(n, dtype=np.int64)
    out_sum    = np.zeros(n, dtype=np.float64)
    out_sum_sq = np.zeros(n, dtype=np.float64)
    out_max_val = np.full(n, -np.inf, dtype=np.float64)
    in_count = np.zeros(n, dtype=np.int64)
    in_sum   = np.zeros(n, dtype=np.float64)
    type_counts = np.zeros((n, len(TRX_TYPES)), dtype=np.int64)

    if is_march:
        march_out_last_ns = np.full(n, -1, dtype=np.int64)
        march_in_last_ns  = np.full(n, -1, dtype=np.int64)
        march_type_last_ns = {t: np.full(n, -1, dtype=np.int64) for t in OUTBOUND_TYPES}
        march_dst_cashin_last_ns = np.full(n, -1, dtype=np.int64)
        march_dst_p2p_last_ns    = np.full(n, -1, dtype=np.int64)
        out_7d_count  = np.zeros(n, dtype=np.int64)
        out_7d_sum    = np.zeros(n, dtype=np.float64)
        out_14d_count = np.zeros(n, dtype=np.int64)
        out_14d_sum   = np.zeros(n, dtype=np.float64)
        in_7d_count   = np.zeros(n, dtype=np.int64)
        in_7d_sum     = np.zeros(n, dtype=np.float64)
        in_14d_count  = np.zeros(n, dtype=np.int64)
        in_14d_sum    = np.zeros(n, dtype=np.float64)
        MARCH_7D_NS  = pd.Timestamp("2024-03-25 00:00:00").value
        MARCH_14D_NS = pd.Timestamp("2024-03-18 00:00:00").value

    columns = ["SRC_ACCOUNT", "DST_ACCOUNT", "TRX_TYPE", "TRX_AMT", "TRX_DATETIME"]
    pf = pq.ParquetFile(filepath)
    batch_num = 0

    for batch in pf.iter_batches(columns=columns):
        chunk = batch.to_pandas()
        batch_num += 1

        dt_ns = pd.to_datetime(chunk["TRX_DATETIME"]).values.astype(np.int64)
        amt = chunk["TRX_AMT"].values.astype(np.float64)
        trx_type_ints = chunk["TRX_TYPE"].map(TYPE_TO_INT).values

        # Source (Outbound)
        src_mapped = chunk["SRC_ACCOUNT"].map(id_to_int)
        src_valid_mask = src_mapped.notna().values
        if src_valid_mask.any():
            src_idx = src_mapped.values[src_valid_mask].astype(np.int64)
            src_amt = amt[src_valid_mask]
            src_dt  = dt_ns[src_valid_mask]
            src_types = trx_type_ints[src_valid_mask]

            np.add.at(out_count, src_idx, 1)
            np.add.at(out_sum, src_idx, src_amt)
            np.add.at(out_sum_sq, src_idx, src_amt ** 2)
            np.maximum.at(out_max_val, src_idx, src_amt)

            valid_type = ~np.isnan(src_types)
            if valid_type.any():
                np.add.at(type_counts,
                          (src_idx[valid_type], src_types[valid_type].astype(np.int64)), 1)

            np.maximum.at(global_last_dt_ns, src_idx, src_dt)

            if is_march:
                np.maximum.at(march_out_last_ns, src_idx, src_dt)
                for t in OUTBOUND_TYPES:
                    t_int = TYPE_TO_INT[t]
                    t_mask = src_types == t_int
                    if t_mask.any():
                        np.maximum.at(march_type_last_ns[t], src_idx[t_mask], src_dt[t_mask])
                mask_7d = src_dt >= MARCH_7D_NS
                if mask_7d.any():
                    np.add.at(out_7d_count, src_idx[mask_7d], 1)
                    np.add.at(out_7d_sum, src_idx[mask_7d], src_amt[mask_7d])
                mask_14d = src_dt >= MARCH_14D_NS
                if mask_14d.any():
                    np.add.at(out_14d_count, src_idx[mask_14d], 1)
                    np.add.at(out_14d_sum, src_idx[mask_14d], src_amt[mask_14d])

        # Destination (Inbound)
        dst_mapped = chunk["DST_ACCOUNT"].map(id_to_int)
        dst_valid_mask = dst_mapped.notna().values
        if dst_valid_mask.any():
            dst_idx = dst_mapped.values[dst_valid_mask].astype(np.int64)
            dst_amt = amt[dst_valid_mask]
            dst_dt  = dt_ns[dst_valid_mask]
            dst_types = trx_type_ints[dst_valid_mask]

            np.add.at(in_count, dst_idx, 1)
            np.add.at(in_sum, dst_idx, dst_amt)
            np.maximum.at(global_last_dt_ns, dst_idx, dst_dt)

            if is_march:
                np.maximum.at(march_in_last_ns, dst_idx, dst_dt)
                cashin_mask = dst_types == TYPE_TO_INT["CashIn"]
                if cashin_mask.any():
                    np.maximum.at(march_dst_cashin_last_ns, dst_idx[cashin_mask], dst_dt[cashin_mask])
                p2p_mask = dst_types == TYPE_TO_INT["P2P"]
                if p2p_mask.any():
                    np.maximum.at(march_dst_p2p_last_ns, dst_idx[p2p_mask], dst_dt[p2p_mask])
                mask_7d = dst_dt >= MARCH_7D_NS
                if mask_7d.any():
                    np.add.at(in_7d_count, dst_idx[mask_7d], 1)
                    np.add.at(in_7d_sum, dst_idx[mask_7d], dst_amt[mask_7d])
                mask_14d = dst_dt >= MARCH_14D_NS
                if mask_14d.any():
                    np.add.at(in_14d_count, dst_idx[mask_14d], 1)
                    np.add.at(in_14d_sum, dst_idx[mask_14d], dst_amt[mask_14d])

        del chunk
        if batch_num % 10 == 0:
            gc.collect()

    # Finalize monthly features
    print(f"    Finalizing {month} features ({batch_num} batches in {time.time()-t0:.1f}s)...")
    monthly = pd.DataFrame(index=pd.RangeIndex(n, name="aid_int"))
    monthly[f"out_count_{month}"] = out_count
    monthly[f"out_sum_{month}"]   = out_sum
    with np.errstate(divide='ignore', invalid='ignore'):
        monthly[f"out_avg_{month}"] = np.where(out_count > 0, out_sum / out_count, 0.0)
        variance = np.where(out_count > 1, (out_sum_sq - (out_sum ** 2) / out_count) / (out_count - 1), 0.0)
        variance = np.maximum(variance, 0.0)
    monthly[f"out_std_{month}"] = np.sqrt(variance)
    monthly[f"out_max_{month}"] = np.where(np.isinf(out_max_val), 0.0, out_max_val)
    monthly[f"in_count_{month}"] = in_count
    monthly[f"in_sum_{month}"]   = in_sum
    for i, ttype in enumerate(TRX_TYPES):
        monthly[f"count_{ttype}_{month}"] = type_counts[:, i]
    monthly.to_parquet(os.path.join(FEATURE_STORE_DIR, f"trx_agg_{month}.parquet"))
    print(f"    ✅ Wrote trx_agg_{month}.parquet ({monthly.shape[1]} columns)")

    # March advanced features
    if is_march:
        print(f"    Finalizing March advanced features...")
        march_adv = pd.DataFrame(index=pd.RangeIndex(n, name="aid_int"))
        march_adv["days_since_last_outbound"] = np.where(march_out_last_ns > 0, np.floor((REF_DATE_MARCH_NS - march_out_last_ns) / NS_PER_DAY), 31.0)
        march_adv["days_since_last_inbound"] = np.where(march_in_last_ns > 0, np.floor((REF_DATE_MARCH_NS - march_in_last_ns) / NS_PER_DAY), 31.0)
        for t in OUTBOUND_TYPES:
            march_adv[f"days_since_last_{t}"] = np.where(march_type_last_ns[t] > 0, np.floor((REF_DATE_MARCH_NS - march_type_last_ns[t]) / NS_PER_DAY), 31.0)
        march_adv["days_since_last_CashIn"] = np.where(march_dst_cashin_last_ns > 0, np.floor((REF_DATE_MARCH_NS - march_dst_cashin_last_ns) / NS_PER_DAY), 31.0)
        march_adv["days_since_received_P2P"] = np.where(march_dst_p2p_last_ns > 0, np.floor((REF_DATE_MARCH_NS - march_dst_p2p_last_ns) / NS_PER_DAY), 31.0)
        march_adv["out_count_last_7d"]  = out_7d_count
        march_adv["out_sum_last_7d"]    = out_7d_sum
        march_adv["out_count_last_14d"] = out_14d_count
        march_adv["out_sum_last_14d"]   = out_14d_sum
        march_adv["in_count_last_7d"]   = in_7d_count
        march_adv["in_sum_last_7d"]     = in_7d_sum
        march_adv["in_count_last_14d"]  = in_14d_count
        march_adv["in_sum_last_14d"]    = in_14d_sum
        march_out_ct = out_count.astype(np.float64)
        march_out_sm = out_sum.copy()
        march_adv["out_count_velocity_7d"]  = out_7d_count  / (march_out_ct + 1e-5)
        march_adv["out_count_velocity_14d"] = out_14d_count / (march_out_ct + 1e-5)
        march_adv["out_sum_velocity_7d"]    = out_7d_sum    / (march_out_sm + 1e-5)
        march_adv["out_sum_velocity_14d"]   = out_14d_sum   / (march_out_sm + 1e-5)
        march_adv.to_parquet(os.path.join(FEATURE_STORE_DIR, "trx_march_advanced.parquet"))
        print(f"    ✅ Wrote trx_march_advanced.parquet ({march_adv.shape[1]} columns)")
        del march_adv, march_out_last_ns, march_in_last_ns, march_type_last_ns
        del march_dst_cashin_last_ns, march_dst_p2p_last_ns
        del out_7d_count, out_7d_sum, out_14d_count, out_14d_sum
        del in_7d_count, in_7d_sum, in_14d_count, in_14d_sum

    del monthly, out_count, out_sum, out_sum_sq, out_max_val, in_count, in_sum, type_counts
    gc.collect()

# Global recency
print("\n  Finalizing global recency...")
recency_days = np.where(global_last_dt_ns > 0, np.floor((REF_DATE_RECENCY_NS - global_last_dt_ns) / NS_PER_DAY), 90.0)
recency_df = pd.DataFrame({"days_since_last_trx": recency_days}, index=pd.RangeIndex(n, name="aid_int"))
recency_df.to_parquet(os.path.join(FEATURE_STORE_DIR, "trx_recency.parquet"))
print(f"  ✅ Wrote trx_recency.parquet")
del global_last_dt_ns, recency_df; gc.collect()

### 1.3 — Balance Features (Streaming)

In [ ]:
# ─── Step 3: Streaming Balance Features ───
print("\n[Step 3/6] Streaming balance features...")
bal_files = sorted(glob.glob(os.path.join(base_path, "dayend_balance", "*.parquet")))
print(f"  Found {len(bal_files)} balance files.")

for month_idx, filepath in enumerate(bal_files):
    if month_idx >= len(MONTHS):
        break
    month = MONTHS[month_idx]
    is_march = (month == "march")
    t0 = time.time()
    print(f"\n  Processing {month} balances ({os.path.basename(filepath)})...")

    bal_count  = np.zeros(n, dtype=np.int64)
    bal_sum    = np.zeros(n, dtype=np.float64)
    bal_sum_sq = np.zeros(n, dtype=np.float64)
    bal_min_val = np.full(n, np.inf, dtype=np.float64)
    bal_max_val = np.full(n, -np.inf, dtype=np.float64)

    if is_march:
        daily_bal = np.zeros((n, 31), dtype=np.float32)

    pf = pq.ParquetFile(filepath)
    columns = ["ACCOUNT_ID", "AVAILABLE_BALANCE"]
    if is_march:
        columns.append("DATE")

    batch_num = 0
    for batch in pf.iter_batches(columns=columns):
        chunk = batch.to_pandas()
        batch_num += 1
        mapped = chunk["ACCOUNT_ID"].map(id_to_int)
        valid_mask = mapped.notna().values
        if not valid_mask.any():
            del chunk; continue

        idx = mapped.values[valid_mask].astype(np.int64)
        bal = chunk["AVAILABLE_BALANCE"].values[valid_mask].astype(np.float64)

        np.add.at(bal_count, idx, 1)
        np.add.at(bal_sum, idx, bal)
        np.add.at(bal_sum_sq, idx, bal ** 2)
        np.minimum.at(bal_min_val, idx, bal)
        np.maximum.at(bal_max_val, idx, bal)

        if is_march:
            day_idx = pd.to_datetime(chunk["DATE"].values[valid_mask]).day - 1
            daily_bal[idx, day_idx] = bal.astype(np.float32)

        del chunk
        if batch_num % 20 == 0:
            gc.collect()

    # Finalize
    print(f"    Finalizing {month} balance stats ({batch_num} batches in {time.time()-t0:.1f}s)...")
    bal_monthly = pd.DataFrame(index=pd.RangeIndex(n, name="aid_int"))
    with np.errstate(divide='ignore', invalid='ignore'):
        mean_bal = np.where(bal_count > 0, bal_sum / bal_count, 0.0)
        variance = np.where(bal_count > 1, (bal_sum_sq - (bal_sum ** 2) / bal_count) / (bal_count - 1), 0.0)
        variance = np.maximum(variance, 0.0)
    bal_monthly[f"mean_bal_{month}"] = mean_bal
    bal_monthly[f"std_bal_{month}"]  = np.sqrt(variance)
    bal_monthly[f"min_bal_{month}"]  = np.where(np.isinf(bal_min_val), 0.0, bal_min_val)
    bal_monthly[f"max_bal_{month}"]  = np.where(np.isinf(bal_max_val), 0.0, bal_max_val)
    bal_monthly.to_parquet(os.path.join(FEATURE_STORE_DIR, f"bal_agg_{month}.parquet"))
    print(f"    ✅ Wrote bal_agg_{month}.parquet")

    if is_march:
        print(f"    Computing March daily balance features...")
        march_bal = pd.DataFrame(index=pd.RangeIndex(n, name="aid_int"))
        daily_f64 = daily_bal.astype(np.float64)
        march_bal["final_balance_march"] = daily_f64[:, 30]
        march_bal["balance_drop_march"] = daily_f64[:, 30] - daily_f64[:, 0]
        t_centered = np.arange(31, dtype=np.float64) - 15.0
        denom = (t_centered ** 2).sum()
        weights = t_centered / denom
        march_bal["balance_trend_march"] = daily_f64 @ weights
        march_bal["zero_balance_days_march"] = (daily_bal < 10.0).sum(axis=1).astype(int)
        march_bal["mean_balance_last_7d_march"] = daily_f64[:, 24:31].mean(axis=1)
        march_bal["mean_balance_last_14d_march"] = daily_f64[:, 17:31].mean(axis=1)
        march_bal["zero_balance_days_last_7d_march"] = (daily_bal[:, 24:31] < 10.0).sum(axis=1).astype(int)
        t_14 = np.arange(14, dtype=np.float64) - 6.5
        denom_14 = (t_14 ** 2).sum()
        weights_14 = t_14 / denom_14
        march_bal["balance_trend_last_14d_march"] = daily_f64[:, 17:31] @ weights_14
        march_bal.to_parquet(os.path.join(FEATURE_STORE_DIR, "bal_march_advanced.parquet"))
        print(f"    ✅ Wrote bal_march_advanced.parquet ({march_bal.shape[1]} columns)")
        del daily_bal, daily_f64, march_bal

    del bal_monthly, bal_count, bal_sum, bal_sum_sq, bal_min_val, bal_max_val
    gc.collect()

### 1.4 — Cross-Month & Balance-Derived Features

In [ ]:
# ─── Step 4: Cross-Month Derived Features ───
print("\n[Step 4/6] Computing cross-month derived features...")

trx_dfs = {}
for month in MONTHS:
    fpath = os.path.join(FEATURE_STORE_DIR, f"trx_agg_{month}.parquet")
    if os.path.exists(fpath):
        trx_dfs[month] = pd.read_parquet(fpath)

cross = pd.DataFrame(index=pd.RangeIndex(n, name="aid_int"))
cross["out_count_total"] = sum(trx_dfs[m][f"out_count_{m}"] for m in trx_dfs)
cross["out_sum_total"]   = sum(trx_dfs[m][f"out_sum_{m}"] for m in trx_dfs)
cross["in_count_total"]  = sum(trx_dfs[m][f"in_count_{m}"] for m in trx_dfs)
cross["in_sum_total"]    = sum(trx_dfs[m][f"in_sum_{m}"] for m in trx_dfs)

if "jan" in trx_dfs and "feb" in trx_dfs:
    cross["trx_count_decay_jan_feb"] = (trx_dfs["feb"]["out_count_feb"] - trx_dfs["jan"]["out_count_jan"]) / (trx_dfs["jan"]["out_count_jan"] + 1)
if "feb" in trx_dfs and "march" in trx_dfs:
    cross["trx_count_decay_feb_march"] = (trx_dfs["march"]["out_count_march"] - trx_dfs["feb"]["out_count_feb"]) / (trx_dfs["feb"]["out_count_feb"] + 1)
if "jan" in trx_dfs and "march" in trx_dfs:
    cross["trx_count_decay_jan_march"] = (trx_dfs["march"]["out_count_march"] - trx_dfs["jan"]["out_count_jan"]) / (trx_dfs["jan"]["out_count_jan"] + 1)

if "march" in trx_dfs:
    cross["march_activity_share"] = trx_dfs["march"]["out_count_march"] / (cross["out_count_total"] + 1e-5)

for ttype in TRX_TYPES:
    cross[f"count_{ttype}_total"] = sum(trx_dfs[m].get(f"count_{ttype}_{m}", 0) for m in trx_dfs)

type_total_cols = [f"count_{t}_total" for t in TRX_TYPES]
cross["trx_type_diversity"] = (cross[type_total_cols] > 0).sum(axis=1).astype(int)

denom = cross["out_count_total"] + 1e-5
cross["merchant_pay_ratio"] = cross["count_MerchantPay_total"] / denom
cross["bill_pay_ratio"]     = cross["count_BillPay_total"] / denom
cross["p2p_ratio"]          = cross["count_P2P_total"] / denom
cross["cashout_ratio"]      = cross["count_CashOut_total"] / denom

if "march" in trx_dfs:
    cross["net_flow_march"] = trx_dfs["march"]["in_sum_march"] - trx_dfs["march"]["out_sum_march"]

cross.to_parquet(os.path.join(FEATURE_STORE_DIR, "trx_cross_month.parquet"))
print(f"  ✅ Wrote trx_cross_month.parquet ({cross.shape[1]} columns)")
del trx_dfs, cross; gc.collect()

# Balance-derived features
print("  Computing balance-derived features...")
bal_jan = pd.read_parquet(os.path.join(FEATURE_STORE_DIR, "bal_agg_jan.parquet"))
bal_feb = pd.read_parquet(os.path.join(FEATURE_STORE_DIR, "bal_agg_feb.parquet"))
bal_mar = pd.read_parquet(os.path.join(FEATURE_STORE_DIR, "bal_agg_march.parquet"))
bal_mar_adv = pd.read_parquet(os.path.join(FEATURE_STORE_DIR, "bal_march_advanced.parquet"))

bal_derived = pd.DataFrame(index=pd.RangeIndex(n, name="aid_int"))
bal_derived["balance_stability_march"] = bal_mar["std_bal_march"] / (bal_mar["mean_bal_march"] + 1e-5)
bal_derived["balance_change_jan_march"] = bal_mar["mean_bal_march"] - bal_jan["mean_bal_jan"]
bal_derived["balance_change_feb_march"] = bal_mar["mean_bal_march"] - bal_feb["mean_bal_feb"]
bal_derived["final_to_mean_balance_ratio_march"] = bal_mar_adv["final_balance_march"] / (bal_mar["mean_bal_march"] + 1e-5)
bal_derived.to_parquet(os.path.join(FEATURE_STORE_DIR, "bal_derived.parquet"))
print(f"  ✅ Wrote bal_derived.parquet ({bal_derived.shape[1]} columns)")
del bal_jan, bal_feb, bal_mar, bal_mar_adv, bal_derived; gc.collect()

### 1.5 — Abstract Features, Sparsity Flags, Log Transforms

In [ ]:
# ─── Step 5: Abstract Features, Sparsity Flags, Log Transforms ───
print("\n[Step 5/6] Computing abstract features, flags, and log transforms...")

cross      = pd.read_parquet(os.path.join(FEATURE_STORE_DIR, "trx_cross_month.parquet"))
recency    = pd.read_parquet(os.path.join(FEATURE_STORE_DIR, "trx_recency.parquet"))
bal_march  = pd.read_parquet(os.path.join(FEATURE_STORE_DIR, "bal_agg_march.parquet"))
bal_mar_adv = pd.read_parquet(os.path.join(FEATURE_STORE_DIR, "bal_march_advanced.parquet"))
march_adv  = pd.read_parquet(os.path.join(FEATURE_STORE_DIR, "trx_march_advanced.parquet"))
march_trx  = pd.read_parquet(os.path.join(FEATURE_STORE_DIR, "trx_agg_march.parquet"))
bal_derived = pd.read_parquet(os.path.join(FEATURE_STORE_DIR, "bal_derived.parquet"))

# Abstract Features
abstract = pd.DataFrame(index=pd.RangeIndex(n, name="aid_int"))
ati = 90.0 / (cross["out_count_total"] + 1)
abstract["recency_pressure"] = recency["days_since_last_trx"] / (ati + 1e-5)
abstract["digital_integration"] = cross["trx_type_diversity"] * (cross["bill_pay_ratio"] + cross["merchant_pay_ratio"] + cross["p2p_ratio"] - cross["cashout_ratio"])
abstract["depletion_velocity"] = cross["trx_count_decay_feb_march"] + (bal_mar_adv["balance_drop_march"] / (bal_march["mean_bal_march"] + 1e-5))
abstract["store_of_value"] = (bal_march["mean_bal_march"] / (cross["in_sum_total"] + 1)) * (1.0 - bal_derived["balance_stability_march"])
abstract["network_stickiness"] = np.log1p(cross["count_P2P_total"]) * (31.0 / (march_adv["days_since_last_P2P"] + march_adv["days_since_received_P2P"] + 1.0))
abstract["wallet_strain"] = march_trx["out_std_march"] / (bal_march["mean_bal_march"] + 1e-5)
abstract.to_parquet(os.path.join(FEATURE_STORE_DIR, "abstract_features.parquet"))
print(f"  ✅ Wrote abstract_features.parquet ({abstract.shape[1]} columns)")

# Sparsity Flags
flags = pd.DataFrame(index=pd.RangeIndex(n, name="aid_int"))
flags["flag_zero_bill_pay"]     = (cross["bill_pay_ratio"] == 0).astype(float)
flags["flag_zero_merchant_pay"] = (cross["merchant_pay_ratio"] == 0).astype(float)
flags["flag_zero_march_trx"]    = (march_trx["out_count_march"] == 0).astype(float)
flags["flag_zero_trx_last_7d_march"] = (march_adv["out_count_last_7d"] == 0).astype(float)
flags.to_parquet(os.path.join(FEATURE_STORE_DIR, "sparsity_flags.parquet"))
print(f"  ✅ Wrote sparsity_flags.parquet")

# Log Transforms
log_sources = {
    "out_sum_total": cross["out_sum_total"], "in_sum_total": cross["in_sum_total"],
    "out_sum_march": march_trx["out_sum_march"], "in_sum_march": march_trx["in_sum_march"],
    "mean_bal_march": bal_march["mean_bal_march"], "std_bal_march": bal_march["std_bal_march"],
    "final_balance_march": bal_mar_adv["final_balance_march"],
    "out_sum_last_7d": march_adv["out_sum_last_7d"], "out_sum_last_14d": march_adv["out_sum_last_14d"],
    "in_sum_last_7d": march_adv["in_sum_last_7d"], "in_sum_last_14d": march_adv["in_sum_last_14d"],
    "mean_balance_last_7d_march": bal_mar_adv["mean_balance_last_7d_march"],
    "mean_balance_last_14d_march": bal_mar_adv["mean_balance_last_14d_march"],
    "network_stickiness": abstract["network_stickiness"], "wallet_strain": abstract["wallet_strain"],
}
log_df = pd.DataFrame(index=pd.RangeIndex(n, name="aid_int"))
for col, series in log_sources.items():
    log_df[f"{col}_log"] = np.log1p(series.clip(lower=0))
log_df.to_parquet(os.path.join(FEATURE_STORE_DIR, "log_transforms.parquet"))
print(f"  ✅ Wrote log_transforms.parquet ({log_df.shape[1]} columns)")

del cross, recency, bal_march, bal_mar_adv, march_adv, march_trx, bal_derived, abstract, flags, log_df
gc.collect()

### 1.6 — Staged Join → Training Table

In [ ]:
# ─── Step 6: Staged Join and Train/Test Split ───
print("\n[Step 6/6] Staged join and train/test split...")

feature_store_files = [
    "kyc_features.parquet", "trx_recency.parquet",
    "trx_agg_jan.parquet", "trx_agg_feb.parquet", "trx_agg_march.parquet",
    "trx_march_advanced.parquet", "trx_cross_month.parquet",
    "bal_agg_jan.parquet", "bal_agg_feb.parquet", "bal_agg_march.parquet",
    "bal_march_advanced.parquet", "bal_derived.parquet",
    "abstract_features.parquet", "sparsity_flags.parquet", "log_transforms.parquet",
]

final = None
for fname in feature_store_files:
    fpath = os.path.join(FEATURE_STORE_DIR, fname)
    if not os.path.exists(fpath):
        print(f"  ⚠️ {fname} not found, skipping.")
        continue
    part = pd.read_parquet(fpath)
    if final is None:
        final = part
    else:
        final = final.join(part, how="left")
        del part
    gc.collect()
    print(f"  Joined {fname} → {final.shape[1]} total columns")

final = final.fillna(0)
final["ACCOUNT_ID"] = final.index.map(int_to_id)
final = final.reset_index(drop=True)

train_ids_set = set(train_sampled["ACCOUNT_ID"])
test_ids_set  = set(test_sampled["ACCOUNT_ID"])

train_features = final[final["ACCOUNT_ID"].isin(train_ids_set)].copy()
train_features = train_features.merge(train_sampled[["ACCOUNT_ID", "CHURN"]], on="ACCOUNT_ID", how="left")
test_features = final[final["ACCOUNT_ID"].isin(test_ids_set)].copy()

del final; gc.collect()

train_features.to_parquet(os.path.join(PROCESSED_DIR, "train_features.parquet"))
test_features.to_parquet(os.path.join(PROCESSED_DIR, "test_features.parquet"))

print(f"\n  ✅ Train features: {train_features.shape}")
print(f"  ✅ Test features: {test_features.shape}")
print(f"  Features saved to {PROCESSED_DIR}/")

del train_features, test_features; gc.collect()
print("\n" + "="*60)
print("  ✅ FEATURE ENGINEERING COMPLETE!")
print("="*60)

---
## 🧠 Step 2: Model Training (6 Models × 10-Fold CV on T4 GPU)

In [ ]:
# ─── Load Engineered Features ───
train_df = pd.read_parquet(os.path.join(PROCESSED_DIR, "train_features.parquet"))
test_df  = pd.read_parquet(os.path.join(PROCESSED_DIR, "test_features.parquet"))

exclude_cols = ["ACCOUNT_ID", "CHURN"]
feature_cols = [col for col in train_df.columns if col not in exclude_cols]

X = train_df[feature_cols].copy()
y = train_df["CHURN"].copy()
X_test = test_df[feature_cols].copy()

# Clean column names for XGBoost compatibility
clean_cols = [str(col).replace("[", "").replace("]", "").replace("<", "").replace(" ", "_") for col in X.columns]
X.columns = clean_cols
X_test.columns = clean_cols

train_account_ids = train_df["ACCOUNT_ID"].copy()
test_account_ids  = test_df["ACCOUNT_ID"].copy()
del train_df, test_df; gc.collect()

cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

print(f"Features: {X.shape[1]} columns")
print(f"Train: {X.shape[0]:,} rows, Test: {X_test.shape[0]:,} rows")
print(f"Churn rate: {y.mean():.4f}")
print(f"CV: {N_SPLITS}-Fold Stratified")

In [ ]:
# ─── Helper: Evaluate predictions ───
def evaluate_predictions(y_true, y_pred_proba, threshold=0.5):
    y_pred = (y_pred_proba >= threshold).astype(int)
    return {
        "AUC": roc_auc_score(y_true, y_pred_proba),
        "Brier": brier_score_loss(y_true, y_pred_proba),
        "F1": f1_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred)
    }

### 2.1 — LightGBM (GPU)

In [ ]:
# ─── LightGBM ───
print("\n=== Training LightGBM (GPU) ===")
lgb_oof = np.zeros(len(X))
lgb_test = np.zeros(len(X_test))
lgb_models = []

scale_pos_weight = (len(y) - y.sum()) / y.sum()
print(f"  scale_pos_weight: {scale_pos_weight:.4f}")

# Detect GPU for LightGBM
lgb_gpu_params = {"n_jobs": -1}
try:
    model_test = lgb.LGBMClassifier(device="gpu", n_estimators=1)
    model_test.fit(np.random.rand(10, 2), np.random.randint(0, 2, 10))
    lgb_gpu_params = {"device": "gpu"}
    print("  ✅ [GPU] LightGBM GPU enabled.")
except Exception:
    print("  ⚠️ [CPU] LightGBM GPU not available, using CPU.")

for fold, (train_idx, val_idx) in enumerate(cv.split(X, y)):
    params = {
        "objective": "binary", "metric": "auc", "boosting_type": "gbdt",
        "n_estimators": 1500, "learning_rate": 0.03, "num_leaves": 63,
        "max_depth": 8, "min_child_samples": 30, "subsample": 0.8,
        "colsample_bytree": 0.8, "scale_pos_weight": scale_pos_weight,
        "random_state": RANDOM_STATE + fold, "verbose": -1
    }
    params.update(lgb_gpu_params)
    model = lgb.LGBMClassifier(**params)
    model.fit(X.iloc[train_idx], y.iloc[train_idx],
              eval_set=[(X.iloc[val_idx], y.iloc[val_idx])],
              callbacks=[lgb.early_stopping(100, verbose=False)])
    lgb_oof[val_idx] = model.predict_proba(X.iloc[val_idx])[:, 1]
    lgb_test += model.predict_proba(X_test)[:, 1] / N_SPLITS
    lgb_models.append(model)

lgb_metrics = evaluate_predictions(y, lgb_oof)
print(f"\n✅ LightGBM CV: AUC={lgb_metrics['AUC']:.5f}, Brier={lgb_metrics['Brier']:.5f}")
gc.collect()

### 2.2 — XGBoost (GPU)

In [ ]:
# ─── XGBoost ───
print("\n=== Training XGBoost (GPU) ===")
xgb_oof = np.zeros(len(X))
xgb_test = np.zeros(len(X_test))
xgb_models = []

# Detect GPU for XGBoost
xgb_gpu_params = {"n_jobs": -1, "tree_method": "hist"}
try:
    clf = xgb.XGBClassifier(device="cuda", n_estimators=1)
    clf.fit(np.random.rand(10, 2), np.random.randint(0, 2, 10))
    xgb_gpu_params = {"device": "cuda", "tree_method": "hist"}
    print("  ✅ [GPU] XGBoost CUDA enabled.")
except Exception:
    print("  ⚠️ [CPU] XGBoost GPU not available, using CPU.")

for fold, (train_idx, val_idx) in enumerate(cv.split(X, y)):
    params = {
        "objective": "binary:logistic", "eval_metric": "auc",
        "n_estimators": 1500, "learning_rate": 0.03, "max_depth": 7,
        "subsample": 0.8, "colsample_bytree": 0.8, "min_child_weight": 5,
        "scale_pos_weight": scale_pos_weight,
        "random_state": RANDOM_STATE + fold, "early_stopping_rounds": 100
    }
    params.update(xgb_gpu_params)
    model = xgb.XGBClassifier(**params)
    model.fit(X.iloc[train_idx], y.iloc[train_idx],
              eval_set=[(X.iloc[val_idx], y.iloc[val_idx])], verbose=False)
    xgb_oof[val_idx] = model.predict_proba(X.iloc[val_idx])[:, 1]
    xgb_test += model.predict_proba(X_test)[:, 1] / N_SPLITS
    xgb_models.append(model)

xgb_metrics = evaluate_predictions(y, xgb_oof)
print(f"\n✅ XGBoost CV: AUC={xgb_metrics['AUC']:.5f}, Brier={xgb_metrics['Brier']:.5f}")
gc.collect()

### 2.3 — CatBoost (GPU)

In [ ]:
# ─── CatBoost ───
print("\n=== Training CatBoost (GPU) ===")
cat_oof = np.zeros(len(X))
cat_test = np.zeros(len(X_test))
cat_models = []

# Detect GPU for CatBoost
cat_gpu_params = {"thread_count": -1}
try:
    model_test = CatBoostClassifier(task_type="GPU", iterations=1)
    model_test.fit(np.random.rand(10, 2), np.random.randint(0, 2, 10), verbose=False)
    cat_gpu_params = {"task_type": "GPU"}
    print("  ✅ [GPU] CatBoost GPU enabled.")
except Exception:
    print("  ⚠️ [CPU] CatBoost GPU not available, using CPU.")

for fold, (train_idx, val_idx) in enumerate(cv.split(X, y)):
    params = {
        "iterations": 1500, "learning_rate": 0.03, "depth": 7,
        "eval_metric": "AUC", "random_seed": RANDOM_STATE + fold,
        "l2_leaf_reg": 5, "early_stopping_rounds": 100,
        "auto_class_weights": "Balanced"
    }
    params.update(cat_gpu_params)
    model = CatBoostClassifier(**params)
    model.fit(X.iloc[train_idx], y.iloc[train_idx],
              eval_set=(X.iloc[val_idx], y.iloc[val_idx]),
              use_best_model=True, verbose=False)
    cat_oof[val_idx] = model.predict_proba(X.iloc[val_idx])[:, 1]
    cat_test += model.predict_proba(X_test)[:, 1] / N_SPLITS
    cat_models.append(model)

cat_metrics = evaluate_predictions(y, cat_oof)
print(f"\n✅ CatBoost CV: AUC={cat_metrics['AUC']:.5f}, Brier={cat_metrics['Brier']:.5f}")
gc.collect()

### 2.4 — Random Forest (CPU)

In [ ]:
# ─── Random Forest ───
print("\n=== Training Random Forest ===")
rf_oof = np.zeros(len(X))
rf_test = np.zeros(len(X_test))
rf_models = []

for fold, (train_idx, val_idx) in enumerate(cv.split(X, y)):
    model = RandomForestClassifier(
        n_estimators=150, max_depth=12,
        random_state=RANDOM_STATE + fold, n_jobs=-1, class_weight="balanced"
    )
    model.fit(X.iloc[train_idx], y.iloc[train_idx])
    rf_oof[val_idx] = model.predict_proba(X.iloc[val_idx])[:, 1]
    rf_test += model.predict_proba(X_test)[:, 1] / N_SPLITS
    rf_models.append(model)

rf_metrics = evaluate_predictions(y, rf_oof)
print(f"\n✅ Random Forest CV: AUC={rf_metrics['AUC']:.5f}, Brier={rf_metrics['Brier']:.5f}")
gc.collect()

### 2.5 — Logistic Regression & MLP (Fold-Internal Scaling)

In [ ]:
# ─── Logistic Regression (fold-internal scaling to prevent leakage) ───
print("\n=== Training Logistic Regression ===")
lr_oof = np.zeros(len(X))
lr_test = np.zeros(len(X_test))
lr_models = []

for fold, (train_idx, val_idx) in enumerate(cv.split(X, y)):
    imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()
    X_train_imp = imputer.fit_transform(X.iloc[train_idx])
    X_val_imp   = imputer.transform(X.iloc[val_idx])
    X_test_imp  = imputer.transform(X_test)
    X_train_scaled = scaler.fit_transform(X_train_imp)
    X_val_scaled   = scaler.transform(X_val_imp)
    X_test_scaled  = scaler.transform(X_test_imp)

    model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE + fold, class_weight="balanced")
    model.fit(X_train_scaled, y.iloc[train_idx])
    lr_oof[val_idx] = model.predict_proba(X_val_scaled)[:, 1]
    lr_test += model.predict_proba(X_test_scaled)[:, 1] / N_SPLITS
    lr_models.append(model)
    del X_train_imp, X_val_imp, X_test_imp, X_train_scaled, X_val_scaled, X_test_scaled
    gc.collect()

lr_metrics = evaluate_predictions(y, lr_oof)
print(f"\n✅ Logistic Regression CV: AUC={lr_metrics['AUC']:.5f}, Brier={lr_metrics['Brier']:.5f}")

In [ ]:
# ─── MLP Classifier (fold-internal scaling) ───
print("\n=== Training MLP Classifier ===")
mlp_oof = np.zeros(len(X))
mlp_test = np.zeros(len(X_test))
mlp_models = []

for fold, (train_idx, val_idx) in enumerate(cv.split(X, y)):
    imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()
    X_train_imp = imputer.fit_transform(X.iloc[train_idx])
    X_val_imp   = imputer.transform(X.iloc[val_idx])
    X_test_imp  = imputer.transform(X_test)
    X_train_scaled = scaler.fit_transform(X_train_imp)
    X_val_scaled   = scaler.transform(X_val_imp)
    X_test_scaled  = scaler.transform(X_test_imp)

    model = MLPClassifier(
        hidden_layer_sizes=(128, 64), activation="relu",
        max_iter=150, alpha=0.001,
        random_state=RANDOM_STATE + fold, early_stopping=True
    )
    model.fit(X_train_scaled, y.iloc[train_idx])
    mlp_oof[val_idx] = model.predict_proba(X_val_scaled)[:, 1]
    mlp_test += model.predict_proba(X_test_scaled)[:, 1] / N_SPLITS
    mlp_models.append(model)
    del X_train_imp, X_val_imp, X_test_imp, X_train_scaled, X_val_scaled, X_test_scaled
    gc.collect()

mlp_metrics = evaluate_predictions(y, mlp_oof)
print(f"\n✅ MLP CV: AUC={mlp_metrics['AUC']:.5f}, Brier={mlp_metrics['Brier']:.5f}")

### 2.6 — Save OOF & Test Predictions + ROC Curves

In [ ]:
# ─── Save OOF and Test Predictions ───
oof_df = pd.DataFrame({
    "ACCOUNT_ID": train_account_ids,
    "CHURN": y,
    "lgb_oof": lgb_oof, "xgb_oof": xgb_oof, "cat_oof": cat_oof,
    "rf_oof": rf_oof, "lr_oof": lr_oof, "mlp_oof": mlp_oof
})
oof_df.to_parquet(os.path.join(PREDICTIONS_DIR, "oof_predictions.parquet"))

test_preds_df = pd.DataFrame({
    "ACCOUNT_ID": test_account_ids,
    "lgb_test": lgb_test, "xgb_test": xgb_test, "cat_test": cat_test,
    "rf_test": rf_test, "lr_test": lr_test, "mlp_test": mlp_test
})
test_preds_df.to_parquet(os.path.join(PREDICTIONS_DIR, "test_predictions.parquet"))

print("✅ OOF and test predictions saved.")

# ─── ROC Curves ───
plt.figure(figsize=(12, 8))
oof_dict = {
    "LightGBM": lgb_oof, "XGBoost": xgb_oof, "CatBoost": cat_oof,
    "Random Forest": rf_oof, "Logistic Regression": lr_oof, "MLP": mlp_oof
}
for model_name, oof_preds in oof_dict.items():
    fpr, tpr, _ = roc_curve(y, oof_preds)
    auc = roc_auc_score(y, oof_preds)
    plt.plot(fpr, tpr, label=f"{model_name} (AUC = {auc:.4f})")
plt.plot([0, 1], [0, 1], "k--", label="Random")
plt.xlabel("FPR"); plt.ylabel("TPR")
plt.title("Model Zoo ROC Curves")
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.savefig(os.path.join(PLOTS_DIR, "roc_curves.png"), dpi=300, bbox_inches="tight")
plt.show()
print("\n✅ Model training complete!")

---
## 🏆 Step 3: Stacking Ensemble + Calibration → predictions.csv

In [ ]:
# ─── Load OOF and Test Predictions ───
oof = pd.read_parquet(os.path.join(PREDICTIONS_DIR, "oof_predictions.parquet"))
test = pd.read_parquet(os.path.join(PREDICTIONS_DIR, "test_predictions.parquet"))

stack_cols = [c for c in oof.columns if c.endswith("_oof")]
test_stack_cols = [c.replace("_oof", "_test") for c in stack_cols]

print("Detected base models:")
for col in stack_cols:
    print(f"  - {col[:-4]}")

X_stack = oof[stack_cols].values
y_ens = oof["CHURN"].values
X_test_stack = test[test_stack_cols].values

results = {}

# Individual Base Model AUCs
print("\n--- Individual Base Model AUCs ---")
individual_aucs = {}
for col in stack_cols:
    auc = roc_auc_score(y_ens, oof[col].values)
    brier = brier_score_loss(y_ens, oof[col].values)
    individual_aucs[col] = auc
    print(f"  {col[:-4]:<20}: AUC = {auc:.5f}, Brier = {brier:.5f}")
    results[f"Base: {col[:-4]}"] = {
        "auc": auc, "brier": brier,
        "oof": oof[col].values, "test": test[col.replace('_oof', '_test')].values
    }

In [ ]:
# ─── Stacking Meta-Classifier ───
cv_ens = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
meta_oof = np.zeros(len(X_stack))
meta_test = np.zeros(len(X_test_stack))

print("Training Stacking Meta-Classifier (Logistic Regression)...")
for fold, (train_idx, val_idx) in enumerate(cv_ens.split(X_stack, y_ens)):
    meta_model = LogisticRegression(max_iter=1000, random_state=42 + fold)
    meta_model.fit(X_stack[train_idx], y_ens[train_idx])
    meta_oof[val_idx] = meta_model.predict_proba(X_stack[val_idx])[:, 1]
    meta_test += meta_model.predict_proba(X_test_stack)[:, 1] / cv_ens.n_splits

iso_meta = IsotonicRegression(out_of_bounds="clip")
iso_meta.fit(meta_oof, y_ens)
calibrated_meta_oof  = iso_meta.predict(meta_oof)
calibrated_meta_test = iso_meta.predict(meta_test)
results["Stacking (Calibrated LR)"] = {
    "auc": roc_auc_score(y_ens, calibrated_meta_oof),
    "brier": brier_score_loss(y_ens, calibrated_meta_oof),
    "oof": calibrated_meta_oof, "test": calibrated_meta_test
}

# ─── Weighted Average ───
total_auc = sum(individual_aucs.values())
weights = {col: auc / total_auc for col, auc in individual_aucs.items()}
weighted_oof = np.zeros(len(X_stack))
weighted_test = np.zeros(len(X_test_stack))
for oof_col, test_col in zip(stack_cols, test_stack_cols):
    w = weights[oof_col]
    weighted_oof += w * oof[oof_col].values
    weighted_test += w * test[test_col].values
results["Weighted Average"] = {
    "auc": roc_auc_score(y_ens, weighted_oof),
    "brier": brier_score_loss(y_ens, weighted_oof),
    "oof": weighted_oof, "test": weighted_test
}

# ─── Rank-Average Blending (Kaggle Standard) ───
print("Computing Rank-Average Blend...")
rank_oof = np.zeros(len(X_stack))
rank_test = np.zeros(len(X_test_stack))
for oof_col, test_col in zip(stack_cols, test_stack_cols):
    w = (individual_aucs[oof_col] ** 2) / sum(auc ** 2 for auc in individual_aucs.values())
    rank_oof += w * (rankdata(oof[oof_col].values) / len(oof))
    rank_test += w * (rankdata(test[test_col].values) / len(test))

iso_rank = IsotonicRegression(out_of_bounds="clip")
iso_rank.fit(rank_oof, y_ens)
calibrated_rank_oof  = iso_rank.predict(rank_oof)
calibrated_rank_test = iso_rank.predict(rank_test)
results["Rank-Average Blend (Calibrated)"] = {
    "auc": roc_auc_score(y_ens, calibrated_rank_oof),
    "brier": brier_score_loss(y_ens, calibrated_rank_oof),
    "oof": calibrated_rank_oof, "test": calibrated_rank_test
}

# ─── Pick Best & Generate Submission ───
best_method = max(results, key=lambda k: results[k]["auc"])
print(f"\n🏆 Best ensemble method: {best_method} (AUC: {results[best_method]['auc']:.5f})")

# Cost-sensitive threshold optimization
best_oof_proba = results[best_method]["oof"]
best_threshold = 0.5
best_cost = float("inf")
for threshold in np.arange(0.1, 0.9, 0.01):
    y_pred = (best_oof_proba >= threshold).astype(int)
    fn = ((y_ens == 1) & (y_pred == 0)).sum()
    fp = ((y_ens == 0) & (y_pred == 1)).sum()
    cost = 5 * fn + 1 * fp
    if cost < best_cost:
        best_cost = cost
        best_threshold = threshold
print(f"Optimal threshold: {best_threshold:.2f} (Expected Loss: {best_cost})")

# Generate predictions.csv
final_test_proba = results[best_method]["test"]
submission = pd.DataFrame({
    "ACCOUNT_ID": test["ACCOUNT_ID"],
    "CHURN_PROB": final_test_proba
})
submission.to_csv("/content/predictions.csv", index=False)
print(f"\n✅ Submission saved to /content/predictions.csv")
print(f"   Shape: {submission.shape}")
print(submission.head())

# Print comparison
print("\n" + "="*60)
print("  ENSEMBLE COMPARISON SUMMARY")
print("="*60)
print(f"{'Method':<35} {'AUC':>8} {'Brier':>8}")
print("-" * 55)
for method, vals in results.items():
    print(f"{method:<35} {vals['auc']:>8.5f} {vals['brier']:>8.5f}")

---
## 💾 Step 4: Download predictions.csv & Copy to Drive

In [ ]:
# Copy predictions.csv to Google Drive for persistence
import shutil
drive_output = "/content/drive/MyDrive/bkash-presents-nsucec-datathon/predictions.csv"
shutil.copy("/content/predictions.csv", drive_output)
print(f"✅ predictions.csv copied to Google Drive: {drive_output}")

# Also offer direct download
from google.colab import files
files.download('/content/predictions.csv')
print("📥 Download triggered!")